# Импорты

In [ ]:
from unittest.mock import patch

import numpy as np
import pandas as pd

import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

import json

from astropy.extern.configobj.validate import is_list
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

from scipy.optimize import minimize

from tqdm.auto import tqdm

# Загрузка датасетов

In [ ]:
krinov = pd.read_csv("data/krinov.csv")
krinov.set_index('wavelength', inplace=True)
krinov = krinov.loc[range(400, 710, 10)]
krinov.head()

In [ ]:
munsell = pd.read_csv("data/munsell.csv")
munsell.set_index('wavelength', inplace=True)
munsell = munsell.loc[range(400, 710, 10)]
munsell.head()

In [ ]:
munsell['0'].T

In [ ]:
sns.set_theme("notebook")
for i in range(0, 1200, 100):
    plt.plot(munsell.index, munsell[str(i)])
plt.show()

In [ ]:
sns.set_theme("notebook")
for i in range(0, 350, 20):
    plt.plot(krinov.index, krinov[str(i)])
plt.show()

In [ ]:
# TODO решить проблемы с colour-science и нормально доставать спектры
# 400-700 нм
light = {
    'D65': np.array([82.75, 91.49, 93.43, 86.68, 104.86, 117.01, 117.81, 114.86, 115.84, 108.81,
                     109.35, 107.80, 104.79, 107.69, 104.41, 104.05, 100.00, 96.33, 95.79, 88.77,
                     90.01, 89.60, 87.70, 83.29, 83.70, 80.03, 80.21, 82.28, 78.28, 69.71,
                     71.61]),

    'A': np.array([14.71, 20.85, 28.66, 38.31, 49.98, 63.72, 79.54, 97.40, 117.31, 139.05,
                    162.53, 187.53, 213.84, 241.14, 269.15, 297.55, 326.04, 354.33, 382.17, 409.30,
                    435.48, 460.50, 484.20, 506.41, 527.01, 545.92, 563.02, 578.26, 591.56, 602.88,
                    612.21])
}

In [ ]:
wavelengths = np.arange(400, 710, 10)

sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

plt.plot(wavelengths, light['D65'], label='D65')
plt.plot(wavelengths, light['A'], label='A')

plt.legend(frameon=True)

plt.xlim(400, 700)
plt.xlabel('wave lengths')
plt.ylim(0, max(light['A']) + 50)

plt.show()

In [ ]:
# Canon_EOS_100D from https://zenodo.org/records/6590768
data = json.load(open('data/Canon_100D.json'))

main_data = data['spectral_data']['data']['main']

sensor = pd.DataFrame.from_dict(main_data, orient='index', columns=['R', 'G', 'B'])

sensor.index = sensor.index.astype(int)
sensor = sensor.sort_index()

sensor.index.name = 'wavelength'

sensor

In [ ]:
sensor = sensor.loc[range(400, 710, 10)]
sensor

In [ ]:
light['D65'].size

In [ ]:
sensor

# Частный пример

In [ ]:
response = sensor.multiply(light['D65'] * krinov['100'], axis=0).sum(axis=0)

In [ ]:
response.values

In [ ]:
patch = np.broadcast_to(response, (24, 24, 3))
patch

In [ ]:
sns.set_theme(style="white")
img_to_show = patch.astype(np.uint8)

img_final = np.clip(img_to_show, 0, 255).astype(np.uint8)
plt.axis('off')
plt.imshow(img_final)
plt.show()

# А теперь делаем сетку патчей как в статье

In [ ]:
krinov.columns

In [ ]:
# (K, N) @ (N, 3) = (K, 3)
selected_surfaces = munsell[[str(i) for i in range(0, 1250, 50)]]
rgb_all = selected_surfaces.T @ (sensor.values * light['D65'][:, np.newaxis])
rgb_all

In [ ]:
rgb_normalized = rgb_all / rgb_all.max()
rgb_8bit = (rgb_normalized * 255).astype(np.uint8)
rgb_8bit.shape

In [ ]:
cells = rgb_8bit.values[:, np.newaxis, np.newaxis, :]

cells_expanded = np.tile(cells, (1, 10, 10, 1))

grid_shape = (5, 5)
grid = cells_expanded.reshape(grid_shape + (10, 10, 3))

final_image = grid.transpose(0, 2, 1, 3, 4).reshape(grid_shape[0]*10, grid_shape[1]*10, 3)

plt.figure(figsize=(10, 8))
plt.imshow(final_image)
plt.axis('off')
plt.show()

# Восстановление D65 по данным rgb из A


In [ ]:
D_65_rgb = munsell.T @ (sensor.values * light['D65'][:, np.newaxis])
A_rgb = munsell.T @ (sensor.values * light['A'][:, np.newaxis])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    D_65_rgb, A_rgb,
    test_size=0.2,
    random_state=179,
    shuffle=True
)

In [ ]:
model = LinearRegression(fit_intercept=False)
model.fit(X_train, y_train)

M_sklearn = model.coef_.T
M_sklearn

# Поиск самого гладкого решения

In [ ]:
def get_gradient_matrix(n=31):
    nabla = np.zeros((n, n))

    # матрица полностью из статьи
    for i in range(1, n-1):
        nabla[i, i] = -1
        nabla[i, i+1] = 1

    val_edge = 1 / np.sqrt(2)
    nabla[0, 0] = -val_edge
    nabla[0, 1] = val_edge

    nabla[-1, -2] = -val_edge
    nabla[-1, -1] = val_edge

    return nabla

nabla = get_gradient_matrix(31)
Q = nabla.T @ nabla

In [ ]:
def estimate_reflectance(rgb_target, sensor_matrix, light_spectrum):
    """
    rgb_target: [R, G, B] - сигнал с камеры
    sensor_matrix: (N, 3) - чувствительность каналов
    light_spectrum: (N,) - спектр источника (например, D65)
    """
    n = len(light_spectrum)

    A = (sensor_matrix.values * light_spectrum[:, np.newaxis]).T

    nabla = get_gradient_matrix(n)
    Q = nabla.T @ nabla
    def objective(s):
        return s.T @ Q @ s

    constraints = [{'type': 'eq', 'fun': lambda s: A @ s - rgb_target}]

    bounds = [(0, 1) for _ in range(n)]

    s0 = np.full(n, 0.5)

    res = minimize(objective, s0, method='SLSQP', bounds=bounds, constraints=constraints)

    return res.x

In [ ]:
recovered_s = estimate_reflectance(y_train.iloc[0], sensor, light['A'])

import matplotlib.pyplot as plt
wavelengths = np.arange(400, 710, 10)
plt.plot(wavelengths, recovered_s, label='Восстановленный')
plt.plot(wavelengths, munsell['923'], label = "изначальный")
plt.ylim(0, 1.1)
plt.legend()
plt.show()

In [ ]:
num_samples = 12
test_indices = range(num_samples)

recovered_list = []
original_list = []

for i in test_indices:
    rec_s = estimate_reflectance(X_test.iloc[i], sensor, light['D65'])
    recovered_list.append(rec_s)

    orig_idx = X_test.index[i]
    original_list.append(munsell[orig_idx])

In [ ]:
rows = 3
cols = 4
fig, axes = plt.subplots(rows, cols, figsize=(15, 10), sharex=True, sharey=True)
axes = axes.flatten()

wavelengths = np.arange(400, 710, 10)

for i in range(num_samples):
    ax = axes[i]
    ax.plot(wavelengths, original_list[i], 'g--', label='Изначальный')
    ax.plot(wavelengths, recovered_list[i], 'r', label='Восстановленный')

    ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

# Пока глупые выводы
- Можно заметить, что спектры хорошо восстановились
- При этом при больших перепадах на концах видно как из-за ограничения на гладкость у нас не получается их повторить

# Цели
- Добавить метрики
- Сравнить при преобразовании матрицей
- Сравнить при несокльких источниках когда ограничений на функцию будет 3 и больше
- Подумать что делать с концами

In [ ]:
illium = pd.read_csv("data/illum.csv")
illium.head()

In [ ]:
illium.set_index('wavelength', inplace=True)
illium = illium.loc[400:700:10]

In [ ]:
illium = illium / illium.sum(axis=0)

In [ ]:
D_65_rgb = munsell.T @ (sensor.values * illium['D65'].values[:, np.newaxis])

munsell_train, munsell_test, D_65_rgb_train, D_65_rgb_test = train_test_split(
    munsell.T, D_65_rgb,
    test_size=0.2,
    random_state=179,
    shuffle=True
)

In [ ]:
models = dict()
for light in illium.columns:
    model = LinearRegression(fit_intercept=False)
    x_train = munsell_train @ (sensor.values * illium[light].values[:, np.newaxis])
    model.fit(x_train, D_65_rgb_train)
    models[light] = model

In [ ]:
num_samples = 12
test_indices = range(num_samples)

recovered_list = []
recovered_list_from_D65 = []
original_list = []

for i in tqdm(test_indices, desc="Processing samples"):
    rec_s_list = []
    rec_from_D65_list = []
    for light in illium.columns:
        rgb = (munsell_test @ (sensor.values * illium[light].values[:, np.newaxis])).iloc[i]
        rec_s = estimate_reflectance(rgb, sensor, illium[light].values)
        rec_from_D65 = estimate_reflectance(models[light].predict(rgb.values.reshape(1, -1))[0], sensor, illium['D65'].values)
        rec_s_list.append(rec_s)
        rec_from_D65_list.append(rec_from_D65)

    recovered_list.append(rec_s_list)
    recovered_list_from_D65.append(rec_from_D65_list)
    original_list.append(munsell_test.iloc[i])

In [ ]:
rows = num_samples # 12
cols = 2
fig, axes = plt.subplots(rows, cols, figsize=(12, 4 * rows), sharex=True, sharey=True)

wavelengths = np.arange(400, 710, 10)

for i in range(num_samples):
    ax_direct = axes[i, 0]
    ax_direct.plot(wavelengths, original_list[i], 'g--', label='Изначальный')
    for rec_s in recovered_list[i]:
        ax_direct.plot(wavelengths, rec_s, 'b', alpha=0.1)
    ax_direct.set_ylim(0, 1.1)

    ax_d65 = axes[i, 1]
    ax_d65.plot(wavelengths, original_list[i], 'g--', label='Изначальный')
    for rec_d65 in recovered_list_from_D65[i]:
        ax_d65.plot(wavelengths, rec_d65, 'r', alpha=0.1)
    ax_d65.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

In [76]:
def calculate_stability_metric(data):
    mu = np.mean(data, axis=1)
    norm_mu = np.linalg.norm(mu, axis=1)
    diffs = data - mu[:, np.newaxis, :]
    norm_diffs = np.linalg.norm(diffs, axis=2)
    errors = norm_diffs / norm_mu[:, np.newaxis]
    return np.mean(errors) * 100

error_direct = calculate_stability_metric(recovered_list)
error_d65 = calculate_stability_metric(recovered_list_from_D65)

print(f"Средняя ошибка стабильности при восстановлении напрямую: {error_direct:.2f}%")
print(f"Средняя ошибка стабильности при восстановлении через d65: {error_d65:.2f}%")

Средняя ошибка стабильности при восстановлении напрямую: 3.39%
Средняя ошибка стабильности при восстановлении через d65: 2.41%
